# Question 19 - 58. Length of Last Word

Given a string `s` consisting of words and spaces, return *the length of the **last** word in the string*.

A **word** is a maximal substring consisting of non-space characters only.

**Example 1:**

    Input: s = "Hello World"
    Output: 5
    Explanation: The last word is "World" with length 5.

**Example 2:**

    Input: s = "   fly me   to   the moon  "
    Output: 4
    Explanation: The last word is "moon" with length 4.

**Example 3:**

    Input: s = "luffy is still joyboy"
    Output: 6
    Explanation: The last word is "joyboy" with length 6.

**Constraints:**

- `1 <= s.length <= 10^4`
- `s` consists of only English letters and spaces `' '`.
- There will be at least one word in `s`.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Reverse scan (skip trailing spaces, then count)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (split) | O(n) | O(n) |
| 2️⃣ Optimized (scan from the end) | O(k + t) | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"I need the length of the last run of non-space characters. Example 2 is the trap: there can be **trailing spaces**, and multiple spaces between words."*

- Leading/trailing/multiple spaces possible? → **Yes.**
- At least one word guaranteed? → **Yes**, so I never return 0 for an all-space string.
- Only letters and spaces (no tabs/newlines)? → **Yes.**

### Step 1 · Brute force

**Idea:** Split the string into words and return the length of the last one. Python's `str.split()` (no argument) already collapses runs of whitespace and drops leading/trailing spaces.

🗣️ *"In Python the one-liner is `len(s.split()[-1])`. It's O(n) time, but it builds a list of every word, so O(n) extra space, just to read the last one. Since all the information I need is at the end of the string, I can work backwards."*

⚠️ *Gotcha:* `s.split(" ")` (with an explicit separator) does **not** collapse spaces. `"moon  ".split(" ")[-1]` is `""`.

In [1]:
class SolutionBrute:
    def lengthOfLastWord(self, s: str) -> int:
        return len(s.split()[-1])

### Step 2 · Optimize

**Bottleneck:** `split` processes the **whole** string and allocates every word, but the answer depends only on the tail.

**Key insight:** Walk from the right end.
1. Skip trailing spaces.
2. Count non-space characters until a space or the start of the string.

That touches only the trailing spaces (`t`) plus the last word (`k`), never the rest.

🗣️ *"I'll scan from the end: first skip trailing spaces, then count letters until I hit a space or the start. It's O(1) extra space, and it only reads the tail of the string. The worst case is still O(n), for a string that's one long word."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| `s.split()[-1]` | ✅ (fine in practice) | Idiomatic and readable, but O(n) extra space and it scans everything. A good first answer, but expect the "no built-ins" follow-up. |
| `s.rstrip().split(" ")[-1]` | ⚠️ | Works, but still allocates, and it's easy to forget the `rstrip`. |
| Forward scan tracking "current word length" | ✅ | O(1) space, but must reset at each space *and* remember the last non-zero length. More state than needed. |
| **Backward scan with an index** | ✅ (best) | O(1) space, reads only the tail, and there's no reset logic. |

In [2]:
class Solution:
    def lengthOfLastWord(self, s: str) -> int:
        i = len(s) - 1
        while i >= 0 and s[i] == " ":     # 1) skip trailing spaces
            i -= 1
        length = 0
        while i >= 0 and s[i] != " ":     # 2) count the last word
            length += 1
            i -= 1
        return length

### Step 3 · Toy example walkthrough

`s = "fly me to the moon  "` (two trailing spaces, length 20)

| phase | i | s[i] | action | length |
|---|---|---|---|---|
| skip | 19 | ' ' | skip | 0 |
| skip | 18 | ' ' | skip | 0 |
| count | 17 | 'n' | count | 1 |
| count | 16 | 'o' | count | 2 |
| count | 15 | 'o' | count | 3 |
| count | 14 | 'm' | count | 4 |
| count | 13 | ' ' | **stop** | **4** ✅ |

We read 7 characters of a 20-character string. Everything before "moon" was never touched.

In [3]:
cases = [
    ("Hello World", 5),
    ("   fly me   to   the moon  ", 4),
    ("luffy is still joyboy", 6),
    ("a", 1),
    ("a ", 1),
    ("   abc", 3),
    ("day", 3),
]
for s, expected in cases:
    assert SolutionBrute().lengthOfLastWord(s) == expected
    assert Solution().lengthOfLastWord(s) == expected
    print(repr(s), "->", expected)
print("All tests passed ✅")

'Hello World' -> 5
'   fly me   to   the moon  ' -> 4
'luffy is still joyboy' -> 6
'a' -> 1
'a ' -> 1
'   abc' -> 3
'day' -> 3
All tests passed ✅


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| `split()` | O(n) | O(n) | Tokenises the whole string |
| **Backward scan** | **O(t + k)** ≤ O(n) | **O(1)** | Only reads the trailing spaces `t` and the last word `k` |

**Complexity, explained:** each character in the tail is visited at most once, by one of the two `while` loops. The worst case (no spaces at all) reads all n characters → O(n). We store one index and one counter → O(1).

**Edge cases:** single letter, trailing spaces, leading spaces, a single word with no spaces.

**Follow-up:** *Unicode whitespace / tabs?* → use `s[i].isspace()` instead of `== " "`.

🗣️ **30-second recap:** *"`split` would work, but it tokenises the whole string. Instead I scan from the end: skip trailing spaces, then count letters until the next space or the start. That's O(1) extra space and it only reads the last word, O(n) in the worst case."*